# 10. Performance Tuning, Optimization & Execution Plans

This is the bridge to advanced PySpark! Understanding how Spark executes queries under the hood allows you to diagnose slow jobs, prevent out-of-memory errors, and build blazing-fast production pipelines.

### Topics Covered:
1. **The Catalyst Optimizer & Tungsten Engine**
2. **Reading and Understanding Execution Plans (`df.explain()`)**
3. **Partition Management: `repartition()` vs `coalesce()`**
4. **Caching & Persistence (`cache`, `persist`, `StorageLevel`, `unpersist`)**
5. **Adaptive Query Execution (AQE) in Modern Spark**
6. **Data Skew & Common Performance Bottlenecks**
7. **Production Optimization Checklist**
8. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("10_Performance_Tuning") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession configured with Adaptive Query Execution (AQE) enabled!")


---
## 2. Under the Hood: The Catalyst Optimizer

When you run a DataFrame query in PySpark, Catalyst transforms your code through 4 phases:

```
Your Python Code
      │
      ▼
1. Unresolved Logical Plan (Checks syntax: are SQL words valid?)
      │
      ▼
2. Analyzed Logical Plan (Consults Catalog: do tables and columns actually exist?)
      │
      ▼
3. Optimized Logical Plan (Applies rule-based optimizations: predicate pushdown, projection pruning)
      │
      ▼
4. Physical Plan (Selects best algorithms: e.g. BroadcastHashJoin vs SortMergeJoin)
      │
      ▼
Tungsten Engine (Generates optimized Java bytecode executed on CPU)
```


---
## 3. Reading Execution Plans with `explain()`

* `df.explain()`: Prints the final Physical Plan.
* `df.explain(True)` / `df.explain(extended=True)`: Prints all 4 plans (Parsed, Analyzed, Optimized, Physical).


In [ ]:
# Sample dataset
df = spark.range(1, 1000000).toDF("id")
filtered = df.filter(col("id") % 2 == 0).withColumn("doubled", col("id") * 2)

print("--- PHYSICAL PLAN ---")
filtered.explain()


In [ ]:
print("--- DETAILED 4-PHASE EXECUTION PLAN ---")
filtered.explain(extended=True)


---
## 4. Partition Management: `repartition()` vs `coalesce()`

Partitions determine how many parallel tasks Spark creates.

### The Big Difference:
* **`repartition(N)`**:
  * Can increase or decrease the number of partitions.
  * Causes a **Full Shuffle** (data is randomly redistributed across nodes over the network).
  * Use when you need to evenly distribute skewed data or increase parallelism.
* **`coalesce(N)`**:
  * Can **ONLY decrease** the number of partitions.
  * **Avoids a Full Shuffle** by simply merging adjacent partitions on the same worker node!
  * Dramatically faster when reducing partitions (e.g. before saving to 1 or 2 output files).


In [ ]:
# Check initial partition count
print("Initial partition count:", df.rdd.getNumPartitions())

# A. repartition(): increases partitions to 8 (Causes Shuffle)
repart_df = df.repartition(8)
print("Partitions after repartition(8):", repart_df.rdd.getNumPartitions())

# B. coalesce(): reduces partitions to 2 (No Full Shuffle - Very fast!)
coalesce_df = repart_df.coalesce(2)
print("Partitions after coalesce(2):", coalesce_df.rdd.getNumPartitions())


---
## 5. Caching and Persistence

### When should you cache?
If a DataFrame is used **multiple times** in downstream actions (e.g. in an iterative machine learning loop, or referenced by 3 different reports), caching it keeps the computed data in RAM so Spark doesn't recompute the entire DAG from scratch!

### `cache()` vs `persist()`:
* **`df.cache()`**: Shortcut for `persist(StorageLevel.MEMORY_AND_DISK)`.
* **`df.persist(level)`**: Allows custom storage levels:
  * `StorageLevel.MEMORY_ONLY`: Stores in RAM as deserialized Java objects.
  * `StorageLevel.MEMORY_AND_DISK`: Stores in RAM; spills to disk if RAM is full.
  * `StorageLevel.DISK_ONLY`: Stores only on disk.
* **`df.unpersist()`**: Manually frees the cached data from memory when you're done!


In [ ]:
# Create an expensive DataFrame calculation
heavy_df = spark.range(1, 500000).toDF("num") \
    .filter(col("num") % 5 == 0) \
    .withColumn("square", col("num") * col("num"))

# Cache in memory
heavy_df.cache()

# First Action: Actually computes the DataFrame and saves it to cache
print("First action (caches data):")
count1 = heavy_df.count()
print(f"Total rows: {count1}")

# Second Action: Fetches directly from cache (much faster!)
print("Second action (reads directly from cache):")
first_row = heavy_df.first()
print(f"First row: {first_row}")

# Check if DataFrame is cached
print("Is DataFrame cached?", heavy_df.is_cached)

# IMPORTANT: Always unpersist when you no longer need the cached DataFrame to free RAM!
heavy_df.unpersist()
print("Unpersisted! Memory released.")
print("Is DataFrame cached now?", heavy_df.is_cached)


---
## 6. Adaptive Query Execution (AQE)

Introduced in Spark 3+, AQE re-optimizes the query plan **at runtime** based on actual intermediate statistics!

### 3 Key Features of AQE:
1. **Dynamically Coalescing Shuffle Partitions**: Merges tiny partitions into larger ones so you don't end up with 200 tiny tasks doing nothing.
2. **Dynamically Switching Join Strategies**: If an intermediate table turns out to be small enough, Spark switches from a slow SortMergeJoin to a fast BroadcastHashJoin on the fly!
3. **Dynamically Handling Data Skew**: Automatically splits skewed, oversized partitions into smaller chunks to prevent one slow task from delaying the whole job.

```python
# To enable AQE in your session:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```


---
## 7. Production Optimization Checklist

1. **Avoid `.collect()` on large datasets**; use `.take(n)` or `.limit(n)` to avoid Driver Out-Of-Memory errors.
2. **Filter early**: Apply `filter()` and `drop()` as close to the data source as possible (push down predicates).
3. **Use Columnar formats (Parquet/Delta)** instead of CSV or JSON.
4. **Broadcast small lookup tables** using `broadcast(small_df)`.
5. **Use `coalesce()` instead of `repartition()`** when reducing partition count.
6. **Prefer built-in Spark SQL functions** over custom Python UDFs.
7. **Always `unpersist()`** cached DataFrames when finished.
8. **Tune `spark.sql.shuffle.partitions`**: Default is 200; for small/medium data on a laptop, reduce to 4-8; for large clusters, increase appropriately.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [ ]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")
